# V2 — Perplexity: generacja + ewaluacja (Colab)

Jeden notebook robi **cały pipeline** dla testu `perplexity` w **jednym ładowaniu GPU** (`--eval-after`):

1. generuje baseline + stego
2. liczy perplexity tym samym modelem (bez drugiego loadu → mniej OOM)

Ustaw listy `MODELS` i `THRESHOLDS` poniżej. Wyniki na Drive:

- `runs/<run>/raw_responses.jsonl`
- `runs/<run>/evaluation/perplexity_results.json`

Wymaga **GPU**. Po każdym runie czyści VRAM przed następnym.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/Steganography_benchmarks_V2')
MODEL_CACHE_DIR = PROJECT_DIR / 'models_cache'
RUNS_DIR = PROJECT_DIR / 'runs'

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_DIR)
os.environ['MODEL_CACHE_DIR'] = str(MODEL_CACHE_DIR)

print('PROJECT_DIR:', PROJECT_DIR)
print('RUNS_DIR:', RUNS_DIR)

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from getpass import getpass
import os

if not os.getenv('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('HF_TOKEN (Llama/Gemma): ')

In [ ]:
# --- KONFIGURACJA ---
MODELS = ['qwen', 'llama', 'gemma']          # lista modeli
THRESHOLDS = [0.0, 0.01, 0.05, 0.1]        # lista progów

TOP_N = 16
MAX_NEW_TOKENS = 512
SKIP_IF_EVALUATED = True     # pomiń, gdy jest evaluation/perplexity_results.json
CONTINUE_ON_ERROR = True     # idź dalej po błędzie pojedynczego runu

import gc
import json
import sys
from pathlib import Path

import torch
from notebook_runner import run_live

TEST = 'perplexity'


def free_gpu() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _th_match(a: float, b: float) -> bool:
    return abs(float(a) - float(b)) < 1e-9


def find_run(runs_dir: Path, model: str, threshold: float) -> Path | None:
    """Najnowszy completed run perplexity dla model+threshold."""
    if not runs_dir.exists():
        return None
    candidates: list[tuple[str, Path]] = []
    for manifest_path in runs_dir.glob('*/manifest.json'):
        try:
            manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        except json.JSONDecodeError:
            continue
        if manifest.get('test') != TEST:
            continue
        if manifest.get('model_key') != model:
            continue
        if not _th_match(manifest.get('threshold', -1), threshold):
            continue
        if manifest.get('status') != 'completed':
            continue
        candidates.append((manifest.get('updated_at', ''), manifest_path.parent))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[0])
    return candidates[-1][1]


def has_eval(run_dir: Path) -> bool:
    return (run_dir / 'evaluation' / 'perplexity_results.json').is_file()


def build_pipeline_cmd(model: str, threshold: float, run_dir: Path | None = None) -> list[str]:
    cmd = [
        sys.executable,
        'generate_responses.py',
        '--test', TEST,
        '--threshold', str(threshold),
        '--top-n', str(TOP_N),
        '--max-new-tokens', str(MAX_NEW_TOKENS),
        '--platform', 'colab',
        '--output-root', 'runs',
        '--eval-after',
    ]
    if run_dir is not None:
        cmd += ['--run-dir', str(run_dir)]
    else:
        cmd += ['--model', model]
    return cmd


jobs = [(m, th) for m in MODELS for th in THRESHOLDS]
print(f'Plan: {len(jobs)} runów perplexity (gen + eval, 1× GPU load)')
print(f'  modele: {MODELS}')
print(f'  progi: {THRESHOLDS}')

ok, skipped, failed, results = [], [], [], []

for index, (model, threshold) in enumerate(jobs, start=1):
    label = f'{model} | th={threshold}'
    print('\n' + '=' * 60)
    print(f'[{index}/{len(jobs)}] {label}')
    print('=' * 60)

    free_gpu()
    run_dir = find_run(RUNS_DIR, model, threshold)

    if SKIP_IF_EVALUATED and run_dir is not None and has_eval(run_dir):
        print(f'Pomijam — już ewaluowany: {run_dir.name}')
        skipped.append(label)
        data = json.loads((run_dir / 'evaluation' / 'perplexity_results.json').read_text())
        results.append({
            'model': model,
            'threshold': threshold,
            'run': run_dir.name,
            'baseline_ppl': data.get('baseline_perplexity'),
            'stego_ppl': data.get('perplexity'),
            'delta': data.get('perplexity_delta'),
        })
        continue

    try:
        if run_dir is None:
            print('Nowy run: generacja + eval (--eval-after)...')
        else:
            print(f'Istniejący RAW ({run_dir.name}): eval-only (--eval-after)...')

        run_live(build_pipeline_cmd(model, threshold, run_dir))
        run_dir = run_dir or find_run(RUNS_DIR, model, threshold)
        if run_dir is None or not has_eval(run_dir):
            raise RuntimeError('Brak perplexity_results.json po zakończeniu pipeline')

        data = json.loads((run_dir / 'evaluation' / 'perplexity_results.json').read_text())
        results.append({
            'model': model,
            'threshold': threshold,
            'run': run_dir.name,
            'baseline_ppl': data.get('baseline_perplexity'),
            'stego_ppl': data.get('perplexity'),
            'delta': data.get('perplexity_delta'),
        })
        ok.append(label)
        print(
            f"OK: baseline={data['baseline_perplexity']:.2f}, "
            f"stego={data['perplexity']:.2f}, "
            f"delta={data['perplexity_delta']:+.2f}"
        )
    except Exception as exc:
        print(f'BŁĄD: {exc}')
        failed.append((label, str(exc)))
        if not CONTINUE_ON_ERROR:
            raise
    finally:
        free_gpu()

print('\n' + '#' * 60)
print('PODSUMOWANIE')
print('#' * 60)
print(f'OK: {len(ok)} | Pominięte: {len(skipped)} | Błędy: {len(failed)}')
if failed:
    print('\nBłędne runy:')
    for label, err in failed:
        print(f'  - {label}: {err}')

In [ ]:
# Tabela wyników perplexity
import json
from pathlib import Path

rows = []
for manifest_path in sorted(Path('runs').glob('*/manifest.json')):
    m = json.loads(manifest_path.read_text(encoding='utf-8'))
    if m.get('test') != 'perplexity':
        continue
    run_dir = manifest_path.parent
    eval_path = run_dir / 'evaluation' / 'perplexity_results.json'
    row = {
        'model': m.get('model_key'),
        'th': m.get('threshold'),
        'status': m.get('status'),
        'run': run_dir.name,
        'eval': eval_path.is_file(),
    }
    if eval_path.is_file():
        d = json.loads(eval_path.read_text())
        row['baseline_ppl'] = d.get('baseline_perplexity')
        row['stego_ppl'] = d.get('perplexity')
        row['delta'] = d.get('perplexity_delta')
    rows.append(row)

rows.sort(key=lambda r: (r['model'], float(r['th'])))
print(f'Runów perplexity: {len(rows)}\n')
print(f"{'model':<6} {'th':<5} {'gen':<12} {'eval':<5} {'baseline':>10} {'stego':>10} {'delta':>10}")
print('-' * 70)
for r in rows:
    if r.get('eval'):
        print(
            f"{r['model']:<6} {r['th']:<5} {r['status']:<12} {'yes':<5} "
            f"{r['baseline_ppl']:10.2f} {r['stego_ppl']:10.2f} {r['delta']:+10.2f}"
        )
    else:
        print(f"{r['model']:<6} {r['th']:<5} {r['status']:<12} {'no':<5}")